In [1]:
import polars as pl
import ast
import numpy as np
import time
import os
from tqdm import tqdm
from multiprocessing import Pool
import itertools
import matplotlib.pyplot as plt
import gc
from collections import Counter
import math
import keras
from scipy import sparse
from keras import models, layers
import keras
from keras.models import load_model
os.environ["POLARS_FMT_MAX_ROWS"] = "100"

In [2]:
def normalize_dataset(dataset: pl.DataFrame):
    bool_cols = [col for col in dataset.columns if dataset[col].dtype == pl.Boolean]
    assert len(bool_cols) == dataset.shape[1] - 1
    dataset = dataset.with_columns(
        [
            pl.col(bool_cols).cast(pl.Int8),  # Convert boolean columns to Float
        ]
    )
    return dataset

test_data_dir = r"D:\Project\TLMN\preprocess\data_validation.parquet"
test_data = pl.read_parquet(test_data_dir)
print(f"The shape of the test dataset is ({test_data.shape[0]:,}; {test_data.shape[1]})")
# test_data_normalized = normalize_dataset(test_data)
MLP_model = load_model(r"models\TLMN_2_players_honestly_new.keras")
X_test = test_data.select(pl.exclude("expected_return")).to_numpy().astype(np.float32)
y_pred = MLP_model.predict(X_test, verbose=1, batch_size=512)
test_data = test_data.with_columns(pl.Series(name="y_pred", values=y_pred.flatten()))

The shape of the test dataset is (11,662,718; 365)
22779/22779 ━━━━━━━━━━━━━━━━━━━━ 106s 5ms/step


In [3]:
# Calculate absolute difference between expected_return and predicted values
test_data = test_data.with_columns(
    pl.col("expected_return")
    .sub(pl.col("y_pred"))
    .abs()
    .alias("abs_diff")
)

print(f"\nAverage absolute difference: {test_data['abs_diff'].mean():.4f}")
print(f"Maximum absolute difference: {test_data['abs_diff'].max():.4f}")
print(f"Minimum absolute difference: {test_data['abs_diff'].min():.4f}")


Average absolute difference: 0.0138
Maximum absolute difference: 0.8298
Minimum absolute difference: 0.0000


In [4]:
test_data["abs_diff"].describe(percentiles=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.97, 0.98, 0.99, 1])

statistic,value
str,f64
"""count""",1.1662718e7
"""null_count""",0.0
"""mean""",0.01376
"""std""",0.013373
"""min""",1.9073e-9
"""0%""",1.9073e-9
"""10%""",0.00136
"""20%""",0.003119
"""30%""",0.005199


In [36]:
# Get the row with maximum absolute difference
max_diff_row = test_data.filter(pl.col("abs_diff") == pl.col("abs_diff").max())

# Get the row with second maximum absolute difference
second_max_diff = test_data.select("abs_diff").unique().sort(by="abs_diff", descending=True)[10,0]
second_max_diff_row = test_data.filter(pl.col("abs_diff") == second_max_diff)

# Print the index of the row with maximum absolute difference
max_diff_idx = (test_data["abs_diff"] == second_max_diff_row["abs_diff"]).arg_max()
print(f"\nIndex of row with maximum absolute difference: {max_diff_idx}")


Index of row with maximum absolute difference: 3892541


In [44]:
from tienlen_libs import Utils, Card
# Convert dict_combo_type_to_int from key:value to value:key
dict_int_to_combo_type = {v: k for k, v in Utils.dict_combo_type_to_int.items()} | {10: 10, 11: 11, 12: 12, 13: 13, 14: 14, 15: 15, 16: 16, 17: 17, 18: 18}
print(dict_int_to_combo_type)

def print_row_info(row):
    player2_card = [i for i in range(52) if row[0, f"player2_card_{i}"] == True]
    player1_action_combo_type = [i for i in range(19) if row[0, f"player1_action_combo_type_{i}"] == True][0]
    player1_action_max_card = [i for i in range(52) if row[0, f"player1_action_max_card_{i}"] == True]
    actions_card_player1 = [i for i in range(52) if row[0, f"actions_card_player1_{i}"] == True]
    actions_card_player2 = [i for i in range(52) if row[0, f"actions_card_player2_{i}"] == True]
    deck_remain = [i for i in range(52) if row[0, f"deck_remain_{i}"] == True]
    action_combo_type = [i for i in range(19) if row[0, f"action_combo_type_{i}"] == True][0]
    action_max_card = [i for i in range(52) if row[0, f"action_max_card_{i}"] == True]

    # Print raw initial player2_card by combining current cards and played cards
    raw_player2_card = player2_card + actions_card_player2
    raw_player2_card.sort()
    print(f"Raw initial player2_card: {raw_player2_card}")
    print(f"Player 2's card: ", end=""); Card.print_pretty_cards(player2_card)
    print(f"Raw player2_card: {player2_card}")
    print(f"Player 1's action: {dict_int_to_combo_type[player1_action_combo_type]}", end=" ")
    Card.print_pretty_cards(player1_action_max_card) if len(player1_action_max_card) > 0 else print("PASS")
    print(f"Raw actions card player 1: {actions_card_player1}")
    print(f"Raw actions card player 2: {actions_card_player2}")
     # print(f"Actions card player 1: ", end=""); Card.print_pretty_cards(actions_card_player1)
    # print(f"Actions card player 2: ", end=""); Card.print_pretty_cards(actions_card_player2)
    print(f"Deck remain: ", end=""); Card.print_pretty_cards(deck_remain)
    for i in range(1, 14):
        if row[0, f'len_player1_card_{i}']:
            print(f"Length of player 1's card: {i}")
            break
    print(f"Action combo type: {dict_int_to_combo_type[action_combo_type]}", end=" ")
    Card.print_pretty_cards(action_max_card) if len(action_max_card) > 0 else print("PASS")
    print(f"Player 2 plays first: {row[0, 'player2_play_first']}")
    print(f"Expected return: {row[0, 'expected_return']}, Predicted return: {row[0, 'y_pred']}")
    
# for idx in range(len(test_data)):
#     if test_data[idx, "len_player1_card"] == 1:
# for idx in range(max_diff_idx - 20, max_diff_idx + 10):
#     if 1:
for idx in range(len(high_diff_rows)):
        print(f"\nRow {idx}:")
        print_row_info(high_diff_rows[idx:idx+1])
# print_row_info(second_max_diff_row)

{0: <ComboType.PASS: 'Pass'>, 1: <ComboType.SINGLE: 'Single'>, 2: <ComboType.SINGLE_DEUCE: 'Single Deuce'>, 3: <ComboType.PAIR: 'Pair'>, 4: <ComboType.PAIR_DEUCE: 'Pair Deuce'>, 5: <ComboType.TRIPLET: 'Triplet'>, 6: <ComboType.QUARTET: 'Quartet'>, 7: <ComboType.DOUBLE_SEQUENCE_THREE: 'Double Sequence Three'>, 8: <ComboType.DOUBLE_SEQUENCE_FOUR: 'Double Sequence Four'>, 9: <ComboType.SEQUENCE: 'Sequence'>, 10: 10, 11: 11, 12: 12, 13: 13, 14: 14, 15: 15, 16: 16, 17: 17, 18: 18}

Row 0:
Raw initial player2_card: [0, 4, 5, 6, 7, 9, 10, 24, 32, 34, 45, 49, 50]
Player 2's card:  [3♠],[4♠],[4♣],[4♦],[4♥],[5♦] 
Raw player2_card: [0, 4, 5, 6, 7, 10]
Player 1's action: ComboType.PASS PASS
Raw actions card player 1: [3, 20, 25, 28, 33, 36, 46, 47]
Raw actions card player 2: [9, 24, 32, 34, 45, 49, 50]
Deck remain:  [3♣],[3♦],[5♠],[5♥],[6♠],[6♣],[6♦],[6♥],[7♠],[7♣],[7♦],[7♥],[8♣],[8♦],[8♥],[9♦],[9♥],[T♣],[T♦],[T♥],[J♥],[Q♣],[Q♦],[Q♥],[K♠],[K♣],[K♦],[K♥],[A♠],[2♠],[2♥] 
Length of player 1's card: 5

In [ ]:
target_player2_cards = [12, 20, 25, 28, 30, 32, 39, 43, 48]
# test_data_dir = r"D:\Project\TLMN\preprocess\data_validation.parquet"
# test_data = pl.read_parquet(test_data_dir)
# from tienlen_libs import Utils, Card
# # Convert dict_combo_type_to_int from key:value to value:key
# dict_int_to_combo_type = {v: k for k, v in Utils.dict_combo_type_to_int.items()} | {10: 10, 11: 11, 12: 12, 13: 13}
# print(dict_int_to_combo_type)

# def print_row_info(row):
#     player2_card = [i for i in range(52) if row[0, f"player2_card_{i}"] == True]
#     player1_action_combo_type = [i for i in range(19) if row[0, f"player1_action_combo_type_{i}"] == True][0]
#     player1_action_max_card = [i for i in range(52) if row[0, f"player1_action_max_card_{i}"] == True]
#     actions_card_player1 = [i for i in range(52) if row[0, f"actions_card_player1_{i}"] == True]
#     actions_card_player2 = [i for i in range(52) if row[0, f"actions_card_player2_{i}"] == True]
#     deck_remain = [i for i in range(52) if row[0, f"deck_remain_{i}"] == True]
#     action_combo_type = [i for i in range(19) if row[0, f"action_combo_type_{i}"] == True][0]
#     action_max_card = [i for i in range(52) if row[0, f"action_max_card_{i}"] == True]

#     # Print raw initial player2_card by combining current cards and played cards
#     raw_player2_card = player2_card + actions_card_player2
#     raw_player2_card.sort()
#     print(f"Raw initial player2_card: {raw_player2_card}")
#     print(f"Player 2's card: ", end=""); Card.print_pretty_cards(player2_card)
#     print(f"Raw player2_card: {player2_card}")
#     print(f"Player 1's action: {dict_int_to_combo_type[player1_action_combo_type]}", end=" ")
#     Card.print_pretty_cards(player1_action_max_card) if len(player1_action_max_card) > 0 else print("PASS")
#     print(f"Raw actions card player 1: {actions_card_player1}")
#     print(f"Raw actions card player 2: {actions_card_player2}")
#     print(f"Actions card player 1: ", end=""); Card.print_pretty_cards(actions_card_player1)
#     print(f"Actions card player 2: ", end=""); Card.print_pretty_cards(actions_card_player2)
#     print(f"Deck remain: ", end=""); Card.print_pretty_cards(deck_remain)
#     for i in range(1, 14):
#         if row[0, f'len_player1_card_{i}']:
#             print(f"Length of player 1's card: {i}")
#             break
#     print(f"Action combo type: {dict_int_to_combo_type[action_combo_type]}", end=" ")
#     Card.print_pretty_cards(action_max_card) if len(action_max_card) > 0 else print("PASS")
#     print(f"Player 2 plays first: {row[0, 'player2_play_first']}")
#     print(f"Expected return: {row[0, 'expected_return']}, Predicted return: {row[0, 'y_pred']}")

for row_idx in range(len(test_data)):
    row = test_data[row_idx:row_idx+1]
    player2_card = [i for i in range(52) if row[0, f"player2_card_{i}"] == True]
    # len_player1_card = row[0, "len_player1_card"]
    player1_action_combo_type = [i for i in range(19) if row[0, f"player1_action_combo_type_{i}"] == True][0]
    # action_combo_type = [i for i in range(19) if row[0, f"action_combo_type_{i}"] == True][0]
    # action_max_card = row[0, "action_max_card"]
    
    if player2_card == target_player2_cards and player1_action_combo_type == 1:
        target_row = row
        print("\nFound matching row!")
        print_row_info(target_row)
        # break

In [46]:
high_diff_rows = test_data.filter(pl.col("abs_diff") > 0.5)
print("\nRows with absolute difference > 0.3:")
pl.Config(tbl_cols=-1)
for i in range(1, 14):
    print(f"len_player1_card = {i}: {high_diff_rows[f'len_player1_card_{i}'].value_counts(sort=True)}")


Rows with absolute difference > 0.3:
len_player1_card = 1: shape: (2, 2)
┌────────────────────┬───────┐
│ len_player1_card_1 ┆ count │
│ ---                ┆ ---   │
│ bool               ┆ u32   │
╞════════════════════╪═══════╡
│ false              ┆ 33    │
│ true               ┆ 20    │
└────────────────────┴───────┘
len_player1_card = 2: shape: (2, 2)
┌────────────────────┬───────┐
│ len_player1_card_2 ┆ count │
│ ---                ┆ ---   │
│ bool               ┆ u32   │
╞════════════════════╪═══════╡
│ false              ┆ 35    │
│ true               ┆ 18    │
└────────────────────┴───────┘
len_player1_card = 3: shape: (2, 2)
┌────────────────────┬───────┐
│ len_player1_card_3 ┆ count │
│ ---                ┆ ---   │
│ bool               ┆ u32   │
╞════════════════════╪═══════╡
│ false              ┆ 47    │
│ true               ┆ 6     │
└────────────────────┴───────┘
len_player1_card = 4: shape: (2, 2)
┌────────────────────┬───────┐
│ len_player1_card_4 ┆ count │
│ ---       

In [ ]:
# Plot distribution of absolute differences using matplotlib
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.hist(test_data['abs_diff'], bins=100, edgecolor='black')
plt.title('Distribution of Absolute Differences between Expected and Predicted Values')
plt.xlabel('Absolute Difference')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Plot distribution of len_player1_card using matplotlib
plt.figure(figsize=(15, 6))
plt.hist(test_data['len_player1_card'], bins=range(1,15), edgecolor='black', align='left')
plt.title('Distribution of Player 1 Card Lengths')
plt.xlabel('Number of Cards')
plt.ylabel('Frequency')
plt.xticks(range(1,14))
plt.grid(True, alpha=0.3)
# Add count labels on top of each bar
counts, bins, _ = plt.hist(test_data['len_player1_card'], bins=range(1,15), edgecolor='black', align='left')
for i, count in enumerate(counts):
    plt.text(bins[i], count, f"{int(count):,}", ha='center', va='bottom')
plt.show()

In [ ]:
import glob
import os

data_3_1_dir = r"D:\Project\TLMN\data_3_1"
data_4_1_dir = r"D:\Project\TLMN\data_4_1"
data_6_4_10_dir = r"D:\Project\TLMN\data_6_4_10"
data_10_7_dir = r"D:\Project\TLMN\data_10_7"
data_13_11_dir = r"D:\Project\TLMN\data_13_11"
data_13_dir = r"D:\Project\TLMN\data_13"
pkl_files_data_3_1 = glob.glob(os.path.join(data_3_1_dir, '**/state_*.pkl'), recursive=True)
print(f"Number of pkl files in data_3_1: {len(pkl_files_data_3_1):,}")
pkl_files_data_4_1 = glob.glob(os.path.join(data_4_1_dir, '**/state_*.pkl'), recursive=True)
print(f"Number of pkl files in data_4_1: {len(pkl_files_data_4_1):,}")
pkl_files_data_6_4_10 = glob.glob(os.path.join(data_6_4_10_dir, '**/state_*.pkl'), recursive=True)
print(f"Number of pkl files in data_6_4_10: {len(pkl_files_data_6_4_10):,}")
pkl_files_data_10_7 = glob.glob(os.path.join(data_10_7_dir, '**/state_*.pkl'), recursive=True)
print(f"Number of pkl files in data_10_7: {len(pkl_files_data_10_7):,}")
pkl_files_data_13_11 = glob.glob(os.path.join(data_13_11_dir, '**/state_*.pkl'), recursive=True)
print(f"Number of pkl files in data_13_11: {len(pkl_files_data_13_11):,}")
pkl_files_data_13 = glob.glob(os.path.join(data_13_dir, '**/state_*.pkl'), recursive=True)
print(f"Number of pkl files in data_13: {len(pkl_files_data_13):,}")